## União dos Dados Mensais e Anuais

Depois de coletados os dados de cada ano separadamente pelos notebooks `Extração_mensal_todos_os_anos` e `Extração_anual_todos_os_anos`, ficamos com um CSV por ano dentro de `CSVs/Mensais` e `CSVs/Anuais`. Este notebook reúne todos esses arquivos em apenas dois CSVs finais, um para os dados mensais e outro para os dados anuais, já ordenados cronologicamente, prontos para serem usados na análise.

 ### Importação da biblioteca Pandas

In [1]:
import glob
import os
import pandas as pd

### Diretórios de entrada e saída

Os CSVs de cada ano ficam separados em `CSVs/Mensais` e `CSVs/Anuais`. O resultado da união é salvo direto na pasta `CSVs/Dados unificados`.

In [ ]:
diretorio_mensais = os.path.join("CSVs", "Mensais")
diretorio_anuais = os.path.join("CSVs", "Anuais")
diretorio_saida = "CSVs/Dados unificados"

### Ordem cronológica dos meses

A coluna `Mês` guarda o nome do mês por extenso (`Janeiro`, `Fevereiro`, ...). Ordenar essa coluna diretamente resultaria numa ordem alfabética, e não cronológica. Por isso criamos um dicionário que mapeia cada mês para o seu número, usado só na hora de ordenar as linhas.

In [3]:
ORDEM_MESES = {
    "Janeiro": 1, "Fevereiro": 2, "Março": 3, "Abril": 4,
    "Maio": 5, "Junho": 6, "Julho": 7, "Agosto": 8,
    "Setembro": 9, "Outubro": 10, "Novembro": 11, "Dezembro": 12,
}

### FUNÇÃO LER_CSV

Essa função lê um CSV de um ano e já deixa ele padronizado antes da união. Alguns arquivos foram escritos com um espaço em branco depois de cada vírgula (tanto no cabeçalho quanto nos valores), enquanto outros não têm esse espaço. O parâmetro `skipinitialspace=True` remove esse espaço já na leitura, e o `strip()` aplicado nos nomes das colunas e nos valores em texto garante que todos os arquivos fiquem no mesmo formato, evitando, por exemplo, que "Windows" e " Windows" sejam tratados como especificações diferentes.

In [4]:
def ler_csv(caminho):
    df = pd.read_csv(caminho, encoding="utf-8", skipinitialspace=True)
    df.columns = [c.strip() for c in df.columns]
    for coluna in df.select_dtypes(include=["object", "str"]).columns:
        df[coluna] = df[coluna].str.strip()
    return df

### União dos arquivos mensais

Percorremos todos os arquivos `dados_mensais_<ano>.csv`, lemos cada um com a função `ler_csv` e concatenamos tudo em um único DataFrame. Depois, ordenamos pelo `Ano` e pela ordem cronológica do `Mês` (usando o dicionário `ORDEM_MESES`)

In [5]:
arquivos_mensais = sorted(glob.glob(os.path.join(diretorio_mensais, "dados_mensais_*.csv")))
arquivos_mensais

['CSVs\\Mensais\\dados_mensais_2019.csv',
 'CSVs\\Mensais\\dados_mensais_2020.csv',
 'CSVs\\Mensais\\dados_mensais_2021.csv',
 'CSVs\\Mensais\\dados_mensais_2022.csv',
 'CSVs\\Mensais\\dados_mensais_2023.csv',
 'CSVs\\Mensais\\dados_mensais_2024.csv',
 'CSVs\\Mensais\\dados_mensais_2025.csv',
 'CSVs\\Mensais\\dados_mensais_2026.csv']

In [6]:
df_mensal_completo = pd.concat(
    (ler_csv(arquivo) for arquivo in arquivos_mensais), ignore_index=True
)

df_mensal_completo["_ordem_mes"] = df_mensal_completo["Mês"].map(ORDEM_MESES)
df_mensal_completo = (
    df_mensal_completo.sort_values(by=["Ano", "_ordem_mes"])
    .drop(columns="_ordem_mes")
    .reset_index(drop=True)
)

df_mensal_completo.info()
df_mensal_completo.head()

<class 'pandas.DataFrame'>
RangeIndex: 17382 entries, 0 to 17381
Data columns (total 6 columns):
 #   Column                   Non-Null Count  Dtype
---  ------                   --------------  -----
 0   Categoria                17382 non-null  str  
 1   Especificação            17301 non-null  str  
 2   Porcentagens             17382 non-null  str  
 3   Incrementos/Decrementos  17382 non-null  str  
 4   Mês                      17382 non-null  str  
 5   Ano                      17382 non-null  int64
dtypes: int64(1), str(5)
memory usage: 814.9 KB


,Categoria,Especificação,Porcentagens,Incrementos/Decrementos,Mês,Ano
0,OS Version,Windows,95.92%,+0.06%,Janeiro,2019
1,OS Version,Windows 10 64 bit,63.77%,-0.02%,Janeiro,2019
2,OS Version,Windows 7 64 bit,26.40%,+0.32%,Janeiro,2019
3,OS Version,Windows 8.1 64 bit,3.43%,-0.10%,Janeiro,2019
4,OS Version,Windows 7,1.52%,-0.09%,Janeiro,2019


* <h3>Salvando o CSV mensal unificado</h3>

In [7]:
df_mensal_completo.to_csv(
    os.path.join(diretorio_saida, "dados_mensais_unificado.csv"),
    index=False,
    encoding="utf-8",
)

### União dos arquivos anuais

O mesmo procedimento é repetido para os arquivos `dados_anuais_<ano>.csv`, com a diferença de que a ordenação final é feita só pela coluna `Ano`, já que esses arquivos não têm a coluna `Mês`.

In [8]:
arquivos_anuais = sorted(glob.glob(os.path.join(diretorio_anuais, "dados_anuais_*.csv")))
arquivos_anuais

['CSVs\\Anuais\\dados_anuais_2019.csv',
 'CSVs\\Anuais\\dados_anuais_2020.csv',
 'CSVs\\Anuais\\dados_anuais_2021.csv',
 'CSVs\\Anuais\\dados_anuais_2022.csv',
 'CSVs\\Anuais\\dados_anuais_2023.csv',
 'CSVs\\Anuais\\dados_anuais_2024.csv',
 'CSVs\\Anuais\\dados_anuais_2025.csv']

In [9]:
df_anual_completo = pd.concat(
    (ler_csv(arquivo) for arquivo in arquivos_anuais), ignore_index=True
)
df_anual_completo = df_anual_completo.sort_values(by=["Ano"]).reset_index(drop=True)

df_anual_completo.info()
df_anual_completo.head()

<class 'pandas.DataFrame'>
RangeIndex: 1379 entries, 0 to 1378
Data columns (total 5 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   Categoria                1379 non-null   str    
 1   Especificação            1375 non-null   str    
 2   Porcentagens             1379 non-null   float64
 3   Incrementos/Decrementos  1379 non-null   float64
 4   Ano                      1379 non-null   int64  
dtypes: float64(2), int64(1), str(2)
memory usage: 54.0 KB


,Categoria,Especificação,Porcentagens,Incrementos/Decrementos,Ano
0,OS Version,Windows,96.86,0.94,2019
1,OS Version,Windows 10 64 bit,61.09,-2.68,2019
2,OS Version,Windows 7 64 bit,33.04,6.64,2019
3,OS Version,Windows 8.1 64 bit,1.79,-1.64,2019
4,OS Version,Windows 7,0.60,-0.92,2019


* <h3>Salvando o CSV anual unificado</h3>

In [10]:
df_anual_completo.to_csv(
    os.path.join(diretorio_saida, "dados_anuais_unificado.csv"),
    index=False,
    encoding="utf-8",
)